# L12 — Project 3: Injury Prevention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yanluo/stem-on-stage-notebooks/blob/main/L12/03_injury_prevention/injury_starter.ipynb)

**Goal:** detect three patterns in dancer-mounted accelerometer data that coaches care about — *hard landings*, *left/right asymmetry*, and *fatigue drift across a session*.

**Ethical framing.** This is **assistive**, not diagnostic. We surface patterns; coaches and dancers make calls. Don't tell anyone they're "injured" based on a Python script.

> **New to pandas / numpy / scipy?** Skim [`L12/00_python_data_tools/python_data_tools_starter.ipynb`](../00_python_data_tools/python_data_tools_starter.ipynb) first — it's a 30-minute tour of every function this notebook uses, with tiny standalone examples.

## Step 0 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Part 1 — Landings on a single ankle

## Step 1 — Load 3 minutes of jumps

The bundled recording is 180 s of continuous accelerometer data, ~90 landings. The last minute simulates fatigue: peaks grow as the dancer tires.

In [ ]:
SAMPLE_URL = "https://raw.githubusercontent.com/yanluo/stem-on-stage-notebooks/main/data/sample-landings.csv"

if IN_COLAB:
    df = pd.read_csv(SAMPLE_URL)
else:
    df = pd.read_csv("../../data/sample-landings.csv")

df["mag"] = np.sqrt(df["x"]**2 + df["y"]**2 + df["z"]**2)
print(df.shape)
df.head()

## Step 2 — Plot magnitude

You'll see a forest of spikes. Each tall spike is one landing.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(df["t"], df["mag"], color="purple", linewidth=0.6)
ax.set_xlabel("time (s)")
ax.set_ylabel("|a| (mg)")
plt.show()

## Step 3 — Peak detection

`scipy.signal.find_peaks` does the heavy lifting: it returns the indices of all local maxima above a height threshold. We need *peak height per landing* — the actual impact.

Use `distance=` to require landings to be at least 1 s apart, so we don't double-count the spike's shoulder.

In [ ]:
peak_idx, _ = find_peaks(df["mag"], height=1500, distance=int(1.0 * 20))
peaks = df.iloc[peak_idx][["t", "mag"]].reset_index(drop=True)
print(f"{len(peaks)} landings detected")
peaks.head()

## Step 4 — Histogram of impact magnitudes

*This is the single most useful chart for a coach.* It tells you the typical landing intensity and how often outliers happen.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(peaks["mag"], bins=20, color="crimson", edgecolor="white")
ax.axvline(peaks["mag"].median(), color="black", linestyle="--", label=f"median = {peaks['mag'].median():.0f}")
ax.set_xlabel("landing peak (mg)")
ax.set_ylabel("count")
ax.legend()
plt.show()

print("median:", peaks["mag"].median())
print("max:   ", peaks["mag"].max())
print("hard-landing cutoff (1.3 × median):", round(peaks["mag"].median() * 1.3))

## Step 5 — Fatigue drift: peak magnitude vs. time

If the dancer is tiring, landings get heavier as the session goes on. Fit a straight line through the peaks. A noticeably *upward* slope is a fatigue signal.

In [ ]:
slope, intercept = np.polyfit(peaks["t"], peaks["mag"], 1)
fit_line = slope * peaks["t"] + intercept

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(peaks["t"], peaks["mag"], s=20, color="crimson", label="landings")
ax.plot(peaks["t"], fit_line, color="black", linestyle="--",
        label=f"fit: {slope:+.2f} mg/s")
ax.set_xlabel("time (s)")
ax.set_ylabel("landing peak (mg)")
ax.legend()
plt.show()

print(f"impact grew by {slope * 60:+.0f} mg per minute")

# Part 2 — Left/right asymmetry

## Step 6 — Two ankles

If you mount a micro:bit on each ankle and broadcast their landings, you can compare left vs. right. A consistent imbalance is a *protective compensation* pattern — coaches want to know about it.

The bundled file already has the per-landing peaks pre-computed (we'd extract them with the same `find_peaks` call from each board's stream).

In [ ]:
PAIR_URL = "https://raw.githubusercontent.com/yanluo/stem-on-stage-notebooks/main/data/sample-landings-asymmetric.csv"

if IN_COLAB:
    pair = pd.read_csv(PAIR_URL)
else:
    pair = pd.read_csv("../../data/sample-landings-asymmetric.csv")

pair["asymmetry"] = pair["mag_right"] / pair["mag_left"]
pair.head()

## Step 7 — Plot left vs. right over time

Two traces on the same axes. If they hug each other, the dancer is balanced. If one consistently sits above the other, that's the pattern.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(pair["t"], pair["mag_left"],  "o-", label="left ankle",  color="steelblue")
ax.plot(pair["t"], pair["mag_right"], "o-", label="right ankle", color="crimson")
ax.set_xlabel("time (s)")
ax.set_ylabel("landing peak (mg)")
ax.legend()
plt.show()

ratio = pair["asymmetry"].mean()
print(f"average right/left ratio: {ratio:.3f}   ({(ratio-1)*100:+.1f}%)")

## What to build next (L13 starting goals)

1. **Personal hard-landing threshold.** Use `1.3 × median` from *this* dancer's first 30 landings as their threshold, not a global number. Coaches care about deviation from a dancer's normal, not from someone else's.
2. **Real-time alerting.** A MakeCode Python program that flashes the lantern red when a landing exceeds the personal threshold. Reuses the L9 peak-detection recipe (read → compare → cooldown).
3. **Multi-axis impact.** Hard sideways landings show in `range_x` or `range_y`, not just `mag`. Build a richer per-landing summary.
4. **Rehearsal report.** matplotlib → PDF: peak histogram, peak-vs-time fit, asymmetry plot, summary stats. One page per session, dated. The coach can flip through these week over week.
5. **Compare to ground truth.** Have a coach call out hard landings during a captured rehearsal. Compare the script's flags to the coach's calls. How close does it get?

## Reflect (homework)

Write 3–4 sentences:
- Which extension will you build first, and why?
- What's one thing that could make this analysis *misleading* for a real dancer?